In [1]:
# import libraries
from pathlib import Path
import pandas as pd
import numpy as np

# setup path to csv file
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path_raw_data = PROJECT_ROOT / "data" / "raw" / "PFAS Sample Sites - Surface Water and Fish Tissue.csv"

# read csv
df_raw = pd.read_csv(path_raw_data)

In [2]:
# first unpivot

# set columns to keep with unpivot
columns_ID = [
    'OBJECTID',
    'COMMENTS',
    'DATE_YEAR'
]

# set columns to unpivot on
columns_unpivot = ['PFOS_MEASURE', 'PFOA_MEASURE']

# unpivot df
df_unpivot = df_raw.melt(
    id_vars=columns_ID,
    value_vars=columns_unpivot
)

df_unpivot.rename(columns={'variable': 'analyte'}, inplace=True)

# sort by objectid
df_unpivot.sort_values(by='OBJECTID', inplace=True)

In [3]:
# first split value columns

# when value column contains multiple values, split into one column per value
df_columns_split = df_unpivot['value'].str.split('/', expand=True)

# join split columns df with unpivoted df
df_unpivot = pd.concat([df_unpivot, df_columns_split], axis=1)

# remove original value column
# df_unpivot.drop(columns='value', inplace=True)
df_unpivot.rename(columns={'value': 'original'}, inplace=True)


In [4]:
# second unpivot

# set columns to keep with unpivot
columns_ID2 = columns_ID + ['analyte', 'original']

# unpivot again based of newly split value columns
df_unpivot = df_unpivot.melt(
    id_vars=columns_ID2,
    value_vars=[0, 1, 2, 3]
)

# remove variable column
df_unpivot.drop(columns='variable', inplace=True)

# remove rows with NaN in value column
df_unpivot.dropna(subset=['value'], inplace=True)

In [5]:
# second split columns

# misc clean up
df_unpivot['value'] = df_unpivot['value'].str.replace('Jul-15 0.66', 'Jul-15: 0.66')
df_unpivot['value'] = df_unpivot['value'].str.replace('Sept', 'Sep')
df_unpivot['value'] = df_unpivot['value'].str.replace('May5-21', 'May-21')

# split value column to separate month-day and result values
df_unpivot[['month-day', 'result']] = df_unpivot['value'].str.split(':', expand=True)

# remove value column
df_unpivot.drop(columns=['value'], inplace=True)

# get result as decimal
df_unpivot['result_num'] = df_unpivot['result'].str.replace('*', '')
df_unpivot['result_num'] = pd.to_numeric(df_unpivot['result_num'], errors='coerce')

# separate flag into column
df_unpivot['flag'] = df_unpivot['result'].str.replace(r'(\d+)', '', regex=True)
df_unpivot['flag'] = df_unpivot['flag'].str.replace('.', '')

# drop result column 
df_unpivot.drop(columns=['result'], inplace=True)

In [6]:
# _____________________________________________________________________________________________
# clean up dates

# get date as one date column
df_unpivot['date'] = df_unpivot['DATE_YEAR'].astype(str) + '-' + df_unpivot['month-day']
df_unpivot['date'] = pd.to_datetime(df_unpivot['date'], format='mixed', yearfirst=True)

# drop extra date columns
df_unpivot.drop(columns=['DATE_YEAR', 'month-day'], inplace=True)



# _____________________________________________________________________________________________
# fix comments and flag columns

# remove the whitespace in flag column
df_unpivot['flag'] = df_unpivot['flag'].str.strip()

# replace blank values with nan
df_unpivot['flag'] = df_unpivot['flag'].replace('', np.nan)

# create boolean mask, true when record has flag
df_unpivot['valid_comment'] = ~df_unpivot['flag'].isna()

# filter comments on boolean mask
df_unpivot['comment'] = np.where(df_unpivot['valid_comment'], df_unpivot['COMMENTS'], np.nan)



# _____________________________________________________________________________________________
# fix specific records

condition1 = (df_unpivot['OBJECTID']==1364211) & (df_unpivot['analyte']=='PFOA_MEASURE')
condition2 = (df_unpivot['OBJECTID']==1364211) & (df_unpivot['analyte']=='PFOS_MEASURE')

df_unpivot['comment'] = np.select(
    [condition1, condition2],
    ['**Sample extract diluted. Result is approximate.', '* indicates value between LOD and LOQ'],
    df_unpivot['comment']
)


# sort df
df_unpivot.sort_values(by=["OBJECTID", "analyte", "date"], inplace=True)

# df_unpivot


In [7]:
df_unpivot.to_csv(r'C:\dev\wi-pfas-dashboard\data\processed\processed2.csv')

